# Data preprocessing 

#### import important libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# import tensorflow as tf

#### import the Dataset

In [2]:
train_df = pd.read_csv("train.csv")

OSError: [Errno 22] Invalid argument

In [ ]:
train_df.head()

In [ ]:
train_df.shape

In [ ]:
pd.set_option("display.max_rows", 200)

#### check for missing data

In [ ]:
train_df.isna().head()

In [ ]:
len(train_df)

In [ ]:
missing_percent = train_df.isna().sum() /len(train_df) * 100 # gives null percentage per col
missing_percent = missing_percent[missing_percent > 0].sort_values(ascending =False) # filters only columns with missing values & sorts them 
missing_percent 

#### shrink the df for simplicity 


In [ ]:
train_df_d = train_df.loc[:,['LotArea', 'SaleCondition','SaleType','OverallQual','YearBuilt','GarageType','SalePrice']]
train_df_d

In [ ]:
train_df_d.isna().sum()

#### 2.Fill remaining categorical nulls with 'None'

In [ ]:
print(train_df_d.select_dtypes(include=[np.object_]).isnull().sum().sort_values(ascending=False)) # calc categorical nulls

In [ ]:
categorical_cols = train_df_d.select_dtypes(include =[np.object_]).columns
categorical_cols

In [ ]:
train_df_d[categorical_cols] = train_df_d[categorical_cols].fillna('None')
train_df_d.isna().sum()

#### 3.Fill numerical nulls (use median or 0 depending on meaning)

In [ ]:
# calc numerical nulls
print(train_df_d.select_dtypes(include=[np.int64,np.float64]).isnull().sum().sort_values(ascending=False)) 

In [ ]:
numerical_cols = train_df_d.select_dtypes(include=[np.int64,np.float64])

In [ ]:
for col in numerical_cols:
    if train_df_d[col].isna().sum() > 0:
        train_df_d[col] = train_df_d[col].fillna(train_df_d[col].median())

In [ ]:
train_df_d.head()

In [ ]:
train_df_d.shape

In [ ]:
train_df_d.columns


## 4.Encoding the Categorical Data to binary

**NOTE** : **_ONEHOTENCODER, ONLY ACCEPTS THE CATEGORICAL COLUMNS.THEN MANUALLY COMBINE IT WITH THE NUMERIC COLUMNS._**

In [ ]:
##Step 1: Separate Target
Y = train_df.iloc[:,-1].values
X_raw= train_df_d.iloc[:,:-1]

In [ ]:
print(Y)
print(X_raw)

In [ ]:
# Step 2: Separate Categorical & Numerical Columns
CatCol = X_raw.select_dtypes(include = 'object').columns
NumCol = X_raw.select_dtypes(exclude = 'object').columns

In [ ]:
X_raw[CatCol].columns

In [ ]:
#Step 3: Encode Only Categorical Features
from sklearn.preprocessing import OneHotEncoder
OHE = OneHotEncoder(sparse_output = False) #Returns array (not sparse matrix)
X_cat = OHE.fit_transform(X_raw[CatCol])
print(X_cat,X_cat.shape)

In [ ]:
# Step 4: Concatenate Numerical and Encoded Categorical
X_num =X_raw.select_dtypes(include = 'number').values
X = np.concatenate([X_num, X_cat], axis=1)
print(X,X.shape)

# VISUALIZE DATASET

In [ ]:
plt.plot(X,Y ,'+')
plt.xlabel("input datapoints")
plt.ylabel("output datapoints")
plt.title("Dataset")
plt.show()

In [ ]:
sns.histplot(Y,kde =True)
plt.suptitle("SalePrice Distribution")
plt.xlabel("SalePice")
plt.show()

In [ ]:
for col in train_df_d.columns.drop('SalePrice'):
    sns.scatterplot(x=train_df_d[col], y=train_df_d['SalePrice'])
    plt.title(f'{col} vs SalePrice')
    plt.show()


# BUILD LR MODEL

#### Training phase

In [ ]:
from sklearn.model_selection import train_test_split 
xtrain , xtest ,ytrain, ytest = train_test_split(X,Y,test_size=0.2,random_state = 42)
from sklearn.linear_model import LinearRegression

In [ ]:
print(xtrain.shape,xtest.shape,ytrain.shape,ytest.shape)


In [ ]:
LR = LinearRegression()# takes only 2d data
LR.fit(xtrain,ytrain)

In [ ]:
LR.intercept_

In [ ]:
LR.coef_

#### testing phase

In [ ]:
y_pred = LR.predict(xtest)

# VISUALISE ACTUAL VS PREDICTED DATA

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))
plt.plot(ytest,label='Actual', marker = 'x')
plt.plot(y_pred, label='Predicted', marker='o')
plt.title('Actual vs Predicted House Prices')
plt.xlabel('Samples')
plt.ylabel('Sale Price')
plt.legend()
plt.grid(True)
plt.savefig("Line_Plot.jpg",dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(ytest, y_pred, alpha=0.5)
plt.plot([ytest.min(), ytest.max()], [ytest.min(), ytest.max()],'r--')  # Diagonal reference line

# (1st arg)	These are X-values — the start and end points on the X-axis
# (2nd arg)	These are Y-values — the start and end points on the Y-axis

plt.xlabel('Actual Sale Price')
plt.ylabel('Predicted Sale Price')
plt.title('Actual vs Predicted Prices')
plt.grid(True)
plt.savefig("Scatter_Plot.jpg",dpi=300)
plt.show()


In [ ]:
print(ytest.min(),ytest.max())

# MODEL EVALUATION

In [ ]:
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

In [ ]:
MSE = mean_squared_error(ytest,y_pred)
MAE = mean_absolute_error(ytest,y_pred)
r2= r2_score(ytest,y_pred)

In [ ]:
print("MSE: {}".format(MSE))
print("MAE:{}".format(MAE))
print("R2 Score:{}".format(r2))

# save the trained model 

In [ ]:
import joblib
joblib.dump(LR,"LR_HPP.pkl") 
joblib.dump(OHE,"encoder.pkl")

# other details

In [ ]:
train_df_d.loc[:,'SaleCondition'].unique()

In [ ]:
train_df_d.loc[:,'SaleType'].unique()

In [ ]:
train_df_d.loc[:,'OverallQual'].unique()

In [ ]:
train_df_d.loc[:,'GarageType'].unique()